# WriteWise — SimpleHTR CRNN Training on IAM Words

Trains a 5-layer CNN + 2-layer BiLSTM + CTC (SimpleHTR architecture) on the IAM Handwriting Words dataset in modern Keras 3 / TensorFlow 2.x.

Exports a lightweight `simplehtr_iam.keras` (~15 MB) model artifact to verify activity prompt matching and prevent worksheet prompt mismatches.

## Hardware Requirement
- Colab runtime: **T4 GPU** (Runtime → Change runtime type → T4 GPU)


## 1. Setup & Environment


In [ ]:
import os
import numpy as np
import tensorflow as tf
import keras
from keras import layers, ops
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)
print("Keras version:", keras.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

np.random.seed(42)
tf.random.set_seed(42)


## 2. Download IAM Words Dataset


In [ ]:
!wget -q https://github.com/sayakpaul/Handwriting-Recognizer-in-Keras/releases/download/v1.0.0/IAM_Words.zip
!unzip -o -qq IAM_Words.zip
!mkdir -p data/words
!tar -xf IAM_Words/words.tgz -C data/words
!mv IAM_Words/words.txt data/
print("Dataset downloaded and extracted to data/")


## 3. Dataset Parsing & Vocabulary


In [ ]:
# Standard fixed 78-character IAM vocabulary
IAM_VOCAB = list("!" + '"' + "#&'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz")
char_to_num = layers.StringLookup(vocabulary=IAM_VOCAB, mask_token=None)
num_to_char = layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), mask_token=None, invert=True)

base_image_path = "data/words"
words_list = []
max_len = 0

with open("data/words.txt", "r") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split(" ")
        if len(parts) >= 9 and parts[1] != "err":
            file_id = parts[0]
            part_a = file_id.split("-")[0]
            part_b = f"{part_a}-{file_id.split('-')[1]}"
            img_rel_path = f"{part_a}/{part_b}/{file_id}.png"
            img_full_path = os.path.join(base_image_path, img_rel_path)
            if os.path.exists(img_full_path) and os.path.getsize(img_full_path) > 0:
                transcription = parts[-1].strip()
                if len(transcription) > 0:
                    words_list.append((img_full_path, transcription))
                    max_len = max(max_len, len(transcription))

print(f"Total valid words: {len(words_list)}")
print(f"Max label length: {max_len}")
print(f"Vocab size: {len(char_to_num.get_vocabulary())}")
print("Vocab:", char_to_num.get_vocabulary())

np.random.shuffle(words_list)
n_total = len(words_list)
n_train = int(0.90 * n_total)
n_val = int(0.05 * n_total)

train_data = words_list[:n_train]
val_data = words_list[n_train:n_train + n_val]
test_data = words_list[n_train + n_val:]
print(f"Train: {len(train_data)}, Val: {len(val_data)}, Test: {len(test_data)}")


## 4. Input Pipeline (Preprocessing & Aspect-Ratio Padding)


In [ ]:
IMG_WIDTH = 128
IMG_HEIGHT = 32
BATCH_SIZE = 64
AUTOTUNE = tf.data.AUTOTUNE

def preprocess_image(img_path):
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    
    # Scale preserving aspect ratio
    h = tf.shape(img)[0]
    w = tf.shape(img)[1]
    scale = tf.cast(IMG_HEIGHT, tf.float32) / tf.cast(tf.maximum(h, 1), tf.float32)
    new_w = tf.cast(tf.minimum(tf.cast(w, tf.float32) * scale, tf.cast(IMG_WIDTH, tf.float32)), tf.int32)
    
    img = tf.image.resize(img, [IMG_HEIGHT, new_w])
    # Pad right side to IMG_WIDTH with 1.0 (white background)
    pad_w = IMG_WIDTH - new_w
    img = tf.pad(img, paddings=[[0, 0], [0, pad_w], [0, 0]], constant_values=1.0)
    
    # Transpose so time (width) is axis 0: (128, 32, 1)
    img = tf.transpose(img, [1, 0, 2])
    return img

def encode_sample(img_path, label):
    img = preprocess_image(img_path)
    label_chars = tf.strings.unicode_split(label, input_encoding="UTF-8")
    label_indices = char_to_num(label_chars)
    label_len = tf.shape(label_indices)[0]
    # Pad label to max_len
    label_indices = tf.pad(
        label_indices,
        [[0, max_len - label_len]],
        constant_values=0
    )
    return {"image": img, "label": label_indices, "label_len": label_len}

def create_dataset(data, is_train=True):
    paths, texts = zip(*data)
    ds = tf.data.Dataset.from_tensor_slices((list(paths), list(texts)))
    ds = ds.map(encode_sample, num_parallel_calls=AUTOTUNE)
    if is_train:
        ds = ds.shuffle(2048)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds

train_ds = create_dataset(train_data, is_train=True)
val_ds = create_dataset(val_data, is_train=False)
test_ds = create_dataset(test_data, is_train=False)


## 5. SimpleHTR Architecture (5 Conv2D + 2 BiLSTM + CTC)


In [ ]:
class CTCLayer(layers.Layer):
    def __init__(self, name=None):
        super().__init__(name=name)
        self.loss_fn = tf.keras.backend.ctc_batch_cost

    def call(self, y_true, y_pred, label_length):
        batch_len = ops.cast(ops.shape(y_true)[0], dtype="int64")
        input_length = ops.cast(ops.shape(y_pred)[1], dtype="int64") * ops.ones(shape=(batch_len, 1), dtype="int64")
        label_length = ops.cast(label_length, dtype="int64")
        if len(ops.shape(label_length)) == 1:
            label_length = ops.expand_dims(label_length, axis=-1)
        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)
        return y_pred

def build_simplehtr_model():
    input_img = layers.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1), name="image", dtype="float32")
    labels = layers.Input(name="label", shape=(None,), dtype="int64")
    label_len = layers.Input(name="label_len", shape=(), dtype="int64")

    # 5 CNN Feature Extraction Blocks
    # Block 1
    x = layers.Conv2D(32, (5, 5), padding="same", kernel_initializer="he_normal")(input_img)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)  # (64, 16)

    # Block 2
    x = layers.Conv2D(64, (5, 5), padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)  # (32, 8)

    # Block 3
    x = layers.Conv2D(128, (3, 3), padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)  # (32, 4)

    # Block 4
    x = layers.Conv2D(128, (3, 3), padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)  # (32, 2)

    # Block 5
    x = layers.Conv2D(256, (3, 3), padding="same", kernel_initializer="he_normal")(x)
    x = layers.BatchNormalization()(x)
    x = layers.LeakyReLU(0.2)(x)
    x = layers.MaxPooling2D(pool_size=(1, 2))(x)  # (32, 1)

    # Reshape for RNN sequence: (None, 32, 256)
    x = layers.Reshape(target_shape=(32, 256))(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)

    # 2 BiLSTM Layers
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.2))(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.2))(x)

    # Softmax output (+1 for CTC blank token at index 79)
    num_classes = len(char_to_num.get_vocabulary()) + 1
    output_logits = layers.Dense(num_classes, activation="softmax", name="dense_output")(x)

    # CTC loss layer
    output = CTCLayer(name="ctc_loss")(labels, output_logits, label_len)

    # Training model
    model = keras.models.Model(inputs=[input_img, labels, label_len], outputs=output, name="simplehtr_crnn")
    # Inference model (only takes image input)
    prediction_model = keras.models.Model(inputs=input_img, outputs=output_logits, name="simplehtr_predictor")

    return model, prediction_model

model, prediction_model = build_simplehtr_model()
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0))
model.summary()


## 6. Training with Early Stopping


In [ ]:
EPOCHS = 25

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1)
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


## 7. Inference & Evaluation on Test Split


In [ ]:
def decode_predictions(pred_probs):
    blank_idx = pred_probs.shape[-1] - 1
    vocab_list = num_to_char.get_vocabulary()
    best_paths = np.argmax(pred_probs, axis=-1)
    
    output_text = []
    for path in best_paths:
        chars = []
        prev_idx = -1
        for idx in path:
            if idx != prev_idx:
                if 0 < idx < len(vocab_list) and idx != blank_idx:
                    char_val = vocab_list[idx]
                    if hasattr(char_val, "decode"):
                        char_val = char_val.decode("utf-8")
                    else:
                        char_val = str(char_val)
                    if char_val != "[UNK]":
                        chars.append(char_val)
                prev_idx = idx
        output_text.append("".join(chars).strip())
    return output_text

# Sample visualization on test set
for batch in test_ds.take(1):
    imgs = batch["image"]
    preds = prediction_model.predict(imgs)
    decoded_texts = decode_predictions(preds)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 6))
    for i, ax in enumerate(axes.flat):
        if i < len(decoded_texts):
            display_img = tf.transpose(imgs[i], [1, 0, 2])[:, :, 0].numpy() * 255
            ax.imshow(display_img, cmap="gray")
            ax.set_title(f"Pred: {decoded_texts[i]}")
            ax.axis("off")
    plt.tight_layout()
    plt.show()


## 8. Export Deployable Model (`simplehtr_iam.keras`)


In [ ]:
os.makedirs("artifacts", exist_ok=True)
export_path = "artifacts/simplehtr_iam.keras"
prediction_model.save(export_path)
print(f"Model exported successfully to {export_path} (Size: {os.path.getsize(export_path) / (1024*1024):.2f} MB)")

# To download locally in Colab:
from google.colab import files
files.download(export_path)
